In [4]:
# ===  Import Libraries ===

# General Data Handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Content-Based Filtering
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import linear_kernel  # Faster than cosine_similarity for TF-IDF


# Collaborative Filtering
# Import the full module so you can check its version
import surprise  
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split


import warnings
import gc
import pickle

import re

In [ ]:
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("sklearn", sklearn.__version__)
print("surprise", surprise.__version__)  

numpy 1.26.4
pandas 2.3.3
sklearn 1.7.2
surprise 1.1.4


In [3]:
movies_df = pd.read_csv(r"C:\Users\Admin\Desktop\PROJECTS\movie-recommendation-system\data\rotten_tomatoes_movies.csv",encoding="latin-1")

reviews_df = pd.read_csv(r"C:\Users\Admin\Desktop\PROJECTS\movie-recommendation-system\data\rotten_tomatoes_movie_reviews.csv",encoding="latin-1")

print(movies_df[['id', 'title']].head(10))

C:\Users\Admin\AppData\Local\Temp\ipykernel_21788\2408427024.py:3: DtypeWarning: Columns (1,11,12,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews_df = pd.read_csv(r"C:\Users\Admin\Desktop\PROJECTS\movie-recommendation-system\data\rotten_tomatoes_movie_reviews.csv",encoding="latin-1")


                               id  \
0              space-zombie-bingo   
1                 the_green_grass   
2                       love_lies   
3            the_sore_losers_1997   
4            dinosaur_island_2002   
5                     adrift_2018   
6  malta-con-huevo-scrambled-beer   
7                kakabakaba-ka-ba   
8                      sundowning   
9            1035316-born_to_kill   

                                              title  
0                               Space Zombie Bingo!  
1                                   The Green Grass  
2                                        Love, Lies  
3                                       Sore Losers  
4                                   Dinosaur Island  
5                                            Adrift  
6                                    Scrambled Beer  
7  Kakabakaba ka ba? (Will Your Heart Beat Faster?)  
8                                        Sundowning  
9                                      Born to Kill 

In [4]:
# =========================================================
# DATA CLEANING + PREPROCESSING + FEATURE ENGINEERING
# =========================================================

# =========================================================
# REMOVE UNNAMED COLUMNS
# =========================================================

movies_df = movies_df.loc[
    :,
    ~movies_df.columns.str.contains("^Unnamed")
]

reviews_df = reviews_df.loc[
    :,
    ~reviews_df.columns.str.contains("^Unnamed")
]

# =========================================================
# REMOVE DUPLICATES
# =========================================================

movies_df = movies_df.drop_duplicates(
    subset=["id"]
)

reviews_df = reviews_df.drop_duplicates(
    subset=["reviewId"]
)

print("Movies Shape :", movies_df.shape)
print("Reviews Shape:", reviews_df.shape)

# =========================================================
# MERGE DATASETS
# =========================================================

df = pd.merge(
    movies_df,
    reviews_df,
    on="id",
    how="inner"
)

print("Merged Shape:", df.shape)

# =========================================================
# KEEP REQUIRED COLUMNS
# =========================================================

essential_cols = [

    "id",
    "title",
    "genre",
    "director",
    "rating",
    "originalLanguage",
    "releaseDateTheaters",
    "audienceScore",
    "tomatoMeter",
    "runtimeMinutes",
    "reviewId",
    "criticName",
    "isTopCritic",
    "originalScore",
    "scoreSentiment"
]

df = df[essential_cols].copy()

# =========================================================
# ID CLEANING
# =========================================================

df["id"] = (
    df["id"]
    .astype(str)
    .str.strip()
)

df = df[
    df["id"] != "nan"
]

# =========================================================
# HANDLE MISSING VALUES
# =========================================================

categorical_cols = [

    "genre",
    "director",
    "rating",
    "originalLanguage",
    "scoreSentiment"
]

df[categorical_cols] = (
    df[categorical_cols]
    .fillna("Unknown")
)

# =========================================================
# TITLE CLEANING
# =========================================================

df["title"] = (
    df["title"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# =========================================================
# GENRE CLEANING
# =========================================================

df["genre"] = (
    df["genre"]
    .astype(str)
    .str.lower()
    .str.replace("|", ",", regex=False)
    .str.strip()
)

# =========================================================
# DIRECTOR CLEANING
# =========================================================

df["director"] = (
    df["director"]
    .astype(str)
    .str.lower()
    .str.replace(" ", "", regex=False)
)

# =========================================================
# LANGUAGE NORMALIZATION
# =========================================================

lang_map = {

    "en": "english",
    "fr": "french",
    "es": "spanish",
    "ja": "japanese"
}

df["originalLanguage"] = (
    df["originalLanguage"]
    .replace(lang_map)
    .astype(str)
    .str.lower()
)

# =========================================================
# CLEAN CRITIC SCORES
# =========================================================

df["originalScore"] = df["originalScore"].replace(

    [
        "fresh",
        "rotten",
        "",
        "NA",
        "N/A"
    ],

    np.nan
)

# =========================================================
# LETTER GRADE CONVERSION
# =========================================================

grade_map = {

    "A+": 100,
    "A": 95,
    "A-": 90,
    "B+": 85,
    "B": 80,
    "B-": 75,
    "C+": 70,
    "C": 65,
    "C-": 60,
    "D+": 55,
    "D": 50,
    "D-": 45,
    "F": 30
}

df["originalScore"] = (
    df["originalScore"]
    .replace(grade_map)
)

# =========================================================
# FRACTION SCORE CONVERTER
# =========================================================

def convert_fraction(val):

    if isinstance(val, str) and "/" in val:

        try:

            num, den = val.split("/")

            return (
                float(num)
                /
                float(den)
            ) * 100

        except:

            return np.nan

    return val

df["originalScore"] = (
    df["originalScore"]
    .apply(convert_fraction)
)

# =========================================================
# NUMERIC COLUMNS
# =========================================================

num_cols = [

    "audienceScore",
    "tomatoMeter",
    "runtimeMinutes",
    "originalScore"
]

for col in num_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    median_value = df[col].median()

    df[col] = (
        df[col]
        .fillna(median_value)
    )

# =========================================================
# DATE FEATURES
# =========================================================

df["releaseDateTheaters"] = pd.to_datetime(
    df["releaseDateTheaters"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Year"] = (
    df["releaseDateTheaters"]
    .dt.year
)

df["Year"] = (
    df["Year"]
    .fillna(1900)
    .astype(int)
)

df["Decade"] = (
    (df["Year"] // 10) * 10
)

print(
    df[
        [
            "title",
            "releaseDateTheaters",
            "Year"
        ]
    ].head()
)

# =========================================================
# REVIEW COUNT
# =========================================================

review_counts = (

    df.groupby("id")
      .size()
      .reset_index(name="review_count")

)

df = pd.merge(

    df,

    review_counts,

    on="id",

    how="left"
)

# =========================================================
# SENTIMENT FEATURE
# =========================================================

sentiment_map = {

    "POSITIVE": 1.0,
    "Positive": 1.0,

    "NEUTRAL": 0.5,
    "Neutral": 0.5,

    "NEGATIVE": 0.0,
    "Negative": 0.0
}

df["sentimentWeight"] = (

    df["scoreSentiment"]
    .map(sentiment_map)
    .fillna(0.5)
)

# =========================================================
# METADATA SOUP
# =========================================================

def create_soup(row):

    return (

        str(row["genre"]) + " " +
        str(row["director"]) + " " +
        str(row["rating"]) + " " +
        str(row["originalLanguage"])

    )

df["soup"] = df.apply(
    create_soup,
    axis=1
)

# =========================================================
# BAYESIAN FUNCTION
# =========================================================

def calculate_bayesian_score(
    scores,
    review_counts,
    percentile=0.60
):

    C = scores.mean()

    m = review_counts.quantile(
        percentile
    )

    return (

        (
            review_counts
            /
            (review_counts + m)
        )
        * scores

        +

        (
            m
            /
            (review_counts + m)
        )
        * C

    )

# =========================================================
# CONTENT DATASET
# =========================================================

# =========================================================
# CONTENT DATASET
# =========================================================

content_df = (

    df.groupby(
        "id",
        as_index=False
    )
    .agg({

        "title": "first",

        "genre": "first",

        "director": "first",

        "rating": "first",

        "originalLanguage": "first",

        "releaseDateTheaters": "first",

        "runtimeMinutes": "mean",

        "audienceScore": "mean",

        "tomatoMeter": "mean",

        "review_count": "max",

        "Year": "first",

        "Decade": "first",

        "sentimentWeight": "mean",

        "soup": "first"

    })

)

content_df = (
    content_df
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

# Release year used by recommender

content_df["releaseYear"] = (
    content_df["Year"]
)

     

# =========================================================
# REMOVE DUPLICATE TITLES
# =========================================================

content_df = (
    content_df
    .drop_duplicates(
        subset=["title"]
    )
    .reset_index(drop=True)
)

# =========================================================
# BAYESIAN SCORES
# =========================================================

content_df["bayesianAudienceScore"] = (

    calculate_bayesian_score(

        content_df["audienceScore"],

        content_df["review_count"]

    )
)

content_df["bayesianTomatoMeter"] = (

    calculate_bayesian_score(

        content_df["tomatoMeter"],

        content_df["review_count"]

    )
)

# =========================================================
# FINAL SCORE
# =========================================================

content_df["finalScore"] = (

    0.6
    *
    content_df["bayesianAudienceScore"]

    +

    0.4
    *
    content_df["bayesianTomatoMeter"]

)

# =========================================================
# CHECK OUTPUT
# =========================================================

print("\nFinal Dataset Shape:", df.shape)

print(
    "\nContent Dataset Shape:",
    content_df.shape
)

print(
    content_df[
        [
            "title",
            "review_count",
            "bayesianAudienceScore",
            "bayesianTomatoMeter",
            "finalScore"
        ]
    ].head()
)

Movies Shape : (142052, 14)
Reviews Shape: (1042574, 6)
Merged Shape: (1042571, 19)
        title releaseDateTheaters  Year
0  love, lies                 NaT  1900
1  love, lies                 NaT  1900
2      adrift          2018-06-01  2018
3      adrift          2018-06-01  2018
4      adrift          2018-06-01  2018

Final Dataset Shape: (1042571, 20)

Content Dataset Shape: (47334, 19)
            title  review_count  bayesianAudienceScore  bayesianTomatoMeter  \
0  009 re: cyborg            13              48.724713            37.883442   
1           8 1/2            57              89.059834            95.345800   
2               1             1              62.109936            70.683628   
3           1 day            16              52.307707            53.308427   
4              10            24              54.625652            68.426180   

   finalScore  
0   44.388205  
1   91.574220  
2   65.539413  
3   52.707995  
4   60.145863  


In [5]:

# =========================================================
# TF-IDF VECTORIZATION
# =========================================================

tfidf = TfidfVectorizer(

    stop_words='english',

    max_features=5000,

    dtype=np.float32

)

tfidf_matrix = tfidf.fit_transform(

    content_df['soup']

)

print(

    "\nTF-IDF Matrix Shape:",

    tfidf_matrix.shape

)

# =========================================================
# COLLABORATIVE DATASET
# =========================================================

collab_df = df[
    [
        'criticName',
        'title',
        'originalScore'
    ]
].copy()

# Remove missing values

collab_df = collab_df.dropna(
    subset=[
        'criticName',
        'title',
        'originalScore'
    ]
)

print(

    "\nCollaborative Dataset Shape:",

    collab_df.shape

)

# =========================================================
# SAMPLE OUTPUTS
# =========================================================

print("\nContent Dataset Sample:\n")

print(

    content_df[
        [
            'title',
            'audienceScore',
            'bayesianAudienceScore',
            'tomatoMeter',
            'bayesianTomatoMeter',
            'review_count'
        ]
    ].head()

)

print("\nCollaborative Dataset Sample:\n")

print(collab_df.head())


TF-IDF Matrix Shape: (47334, 5000)

Collaborative Dataset Shape: (1042571, 3)

Content Dataset Sample:

            title  audienceScore  bayesianAudienceScore  tomatoMeter  \
0  009 re: cyborg           43.0              48.724713         23.0   
1           8 1/2           92.0              89.059834         98.0   
2               1           68.0              62.109936         74.0   
3           1 day           49.0              52.307707         47.0   
4              10           53.0              54.625652         68.0   

   bayesianTomatoMeter  review_count  
0            37.883442            13  
1            95.345800            57  
2            70.683628             1  
3            53.308427            16  
4            68.426180            24  

Collaborative Dataset Sample:

      criticName       title  originalScore
0    James Mudge  love, lies           70.0
1     Diva Velez  love, lies           70.0
2    Josh Parham      adrift           70.0
3  Cory Woodroof    

In [6]:

print("\n df Dataset Sample:\n")
print(df[['id', 'title','reviewId']].head(10))

print("\nContent Dataset Sample:\n")
print(content_df[['id', 'title']].head(10))

# Count total valid (non-null) entries in the director column
total_data = content_df['director'].count()

print(f"Total entries present: {total_data}")

print("\nCollaborative Dataset Sample:\n")
print(collab_df.head(10))

print(df.columns)
print("\n**********************\n")
print(content_df.columns)
print(collab_df.columns)


 df Dataset Sample:

            id       title   reviewId
0    love_lies  love, lies    2739073
1    love_lies  love, lies    2333658
2  adrift_2018      adrift  102694850
3  adrift_2018      adrift  102654799
4  adrift_2018      adrift    2816011
5  adrift_2018      adrift    2812722
6  adrift_2018      adrift    2772153
7  adrift_2018      adrift    2759534
8  adrift_2018      adrift    2740870
9  adrift_2018      adrift    2718979

Content Dataset Sample:

                                id                   title
0                    009_re_cyborg          009 re: cyborg
1                           08-Dec                   8 1/2
2                                1                       1
3                            1-day                   1 day
4                               10                      10
5             1000013_12_angry_men            12 angry men
6                     10000292-rat                     rat
7                10000594-guardian                guardian
8  

In [7]:
# ========================================================= with title logic
# HYBRID MOVIE RECOMMENDATION SYSTEM
# TITLE + GENRE + DIRECTOR SEARCH
# WITH FILTERS + SORTING
# =========================================================


# =========================================================
# DATA PREPROCESSING
# =========================================================

# Convert columns to string
text_cols = [
    'title',
    'genre',
    'director',
    'rating',
    'originalLanguage'
]

for col in text_cols:
    content_df[col] = content_df[col].astype(str)

# Lowercase important columns
content_df['title'] = content_df['title'].str.lower()
content_df['genre'] = content_df['genre'].str.lower()
content_df['director'] = content_df['director'].str.lower()
content_df['originalLanguage'] = content_df['originalLanguage'].str.lower()

# =========================================================
# RELEASE YEAR
# =========================================================

content_df['releaseYear'] = pd.to_datetime(
    content_df['releaseDateTheaters'],
    errors='coerce'
).dt.year

content_df['releaseYear'] = (
    content_df['releaseYear']
    .fillna(1900)
    .astype(int)
)

# =========================================================
# RATING MAPPING
# =========================================================

rating_map = {
    'G': 0,
    'TVG': 0,
    'PG': 1,
    'TVPG': 1,
    'PG-13': 2,
    'TV-14': 2,
    'R': 3,
    'TVMA': 3,
    'NC-17': 4,
    'TUY7': 5
}

content_df['rating_numeric'] = (
    content_df['rating']
    .str.upper()
    .map(rating_map)
)

content_df['rating_numeric'] = (
    content_df['rating_numeric']
    .fillna(-1)
)

content_df['rating_label'] = (
    content_df['rating']
    .fillna('Unknown')
)

# =========================================================
# DROPDOWN VALUES
# =========================================================

rating_dropdown = sorted(
    content_df['rating_label']
    .dropna()
    .unique()
    .tolist()
)

language_dropdown = sorted(
    content_df['originalLanguage']
    .dropna()
    .unique()
    .tolist()
)

director_dropdown = sorted(
    content_df['director']
    .dropna()
    .unique()
    .tolist()
)

# =========================================================
# COMBINED FEATURES
# =========================================================

content_df['combined_features'] = (
    content_df['genre'].astype(str) + ' ' +
    content_df['director'].astype(str) + ' ' +
    content_df['rating'].astype(str) + ' ' +
    content_df['originalLanguage'].astype(str) + ' ' +
    content_df['title'].astype(str)
)

# =========================================================
# TF-IDF MATRIX
# =========================================================

tfidf = TfidfVectorizer(
    stop_words='english',
    dtype=np.float32
)

tfidf_matrix = tfidf.fit_transform(
    content_df['combined_features']
)

# =========================================================
# COLLABORATIVE FILTERING
# =========================================================

reader = Reader(rating_scale=(0, 100))

data = Dataset.load_from_df(
    collab_df[['criticName', 'title', 'originalScore']],
    reader
)

trainset = data.build_full_trainset()

svd_model = SVD(
    n_factors=50,
    random_state=42
)

svd_model.fit(trainset)

#===========================
#Filter Function
#=======================
def apply_filters(
    df,
    rating=None,
    language=None,
    year_min=1900,
    year_max=2025,
    director=None
):

    filtered_df = df.copy()

    if rating and rating != "All":

        filtered_df = filtered_df[
            filtered_df["rating"]
            .str.upper()
            == rating.upper()
        ]

    if language and language != "All":

        filtered_df = filtered_df[
            filtered_df["originalLanguage"]
            .str.contains(
                language,
                case=False,
                na=False
            )
        ]

    filtered_df = filtered_df[
        (filtered_df["releaseYear"] >= year_min)
        &
        (filtered_df["releaseYear"] <= year_max)
    ]

    if director and director != "All":

        filtered_df = filtered_df[
            filtered_df["director"]
            .str.contains(
                director,
                case=False,
                na=False
            )
        ]

    return filtered_df

# =========================================================
# SORT RESULTS
# =========================================================

def sort_results(
    df,
    sort_by=None,
    ascending=False
):

    result = df.copy()

    if sort_by == "Audience Score":

        result = result.sort_values(
            by="bayesianAudienceScore",
            ascending=ascending
        )

    elif sort_by == "Tomato Meter":

        result = result.sort_values(
            by="bayesianTomatoMeter",
            ascending=ascending
        )

    elif sort_by == "Release Year":

        result = result.sort_values(
            by="releaseYear",
            ascending=ascending
        )

    elif sort_by == "Hybrid Score":

        result = result.sort_values(
            by="hybrid_score",
            ascending=ascending
        )

    return result.reset_index(drop=True)

# =========================================================
# HYBRID RECOMMENDER FUNCTION
# =========================================================

def integrated_movie_recommender(
    search_input,
    top_n=10,
    rating=None,
    language=None,
    year_min=1900,
    year_max=2025,
    director_filter=None,
    sort_by=None,
    ascending=False
):

    query_clean = str(search_input).lower().strip()

    candidates = content_df.copy()

    # =====================================================
    # APPLY FILTERS FIRST
    # =====================================================

    candidates = apply_filters(
        df=candidates,
        rating=rating,
        language=language,
        year_min=year_min,
        year_max=year_max,
        director=director_filter
    )

    # =====================================================
    # SEARCH TYPE DETECTION
    # =====================================================

    is_title = query_clean in candidates["title"].values

    is_genre = candidates["genre"].str.contains(
        query_clean,
        case=False,
        na=False
    ).any()

    is_director = candidates["director"].str.contains(
        query_clean,
        case=False,
        na=False
    ).any()

    print("\n" + "=" * 70)
    print(f"🔍 SEARCH QUERY : {search_input}")

    top_result = None

    # =====================================================
    # TITLE SEARCH
    # =====================================================

    if is_title:

        print("📌 SEARCH TYPE : TITLE")

        top_result = candidates[
            candidates["title"] == query_clean
        ].head(1)

        idx = content_df[
            content_df["title"] == query_clean
        ].index[0]

        target_vector = tfidf_matrix[idx]

        sim_scores = linear_kernel(
            target_vector,
            tfidf_matrix
        ).flatten()

        candidates["content_score"] = sim_scores[
            candidates.index
        ]

        candidates = candidates[
            candidates["title"] != query_clean
        ]

    # =====================================================
    # GENRE SEARCH
    # =====================================================

    elif is_genre:

        print("📌 SEARCH TYPE : GENRE")

        candidates["content_score"] = candidates["genre"].apply(
            lambda x:
            1.0 if query_clean in str(x).lower()
            else 0.0
        )

    # =====================================================
    # DIRECTOR SEARCH
    # =====================================================

    elif is_director:

        print("📌 SEARCH TYPE : DIRECTOR")

        candidates["content_score"] = candidates["director"].apply(
            lambda x:
            1.0 if query_clean in str(x).lower()
            else 0.0
        )

    # =====================================================
    # FALLBACK
    # =====================================================

    else:

        print("⚠️ NO EXACT MATCH FOUND")

        candidates["content_score"] = 0.1

    # =====================================================
    # COLLABORATIVE SCORE
    # =====================================================

    target_profile = "Top Critic"

    candidates["svd_score"] = candidates["title"].apply(
        lambda movie:
        svd_model.predict(
            target_profile,
            movie
        ).est
    )

    min_score = candidates["svd_score"].min()
    max_score = candidates["svd_score"].max()

    if max_score != min_score:

        candidates["svd_score_norm"] = (
            candidates["svd_score"] - min_score
        ) / (
            max_score - min_score
        )

    else:

        candidates["svd_score_norm"] = 1.0

    # =====================================================
    # HYBRID SCORE
    # =====================================================

    candidates["hybrid_score"] = (
        0.6 * candidates["content_score"]
        +
        0.4 * candidates["svd_score_norm"]
    )

    # =====================================================
    # TOP-N BEST RECOMMENDATIONS
    # =====================================================

    final_output = (
        candidates
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_n)
        .copy()
    )

    # =====================================================
    # DISPLAY SORTING ONLY
    # =====================================================

    final_output = sort_results(
        final_output,
        sort_by=sort_by,
        ascending=ascending
    )

    # =====================================================
    # DISPLAY
    # =====================================================

    if is_title:

        print("\n🎬 TOP RESULT")
        print("-" * 70)

        print(
            top_result[
                [
                    "title",
                    "genre",
                    "director",
                    "rating",
                    "originalLanguage",
                    "releaseYear"
                ]
            ].to_string(index=False)
        )

        print("\n🔥 HYBRID FILTERED RESULTS ONLY")
        print("-" * 70)

    else:

        print("\n🔥 HYBRID FILTERED RESULTS ONLY")
        print("-" * 70)

    print(
        final_output[
            [
                "title",
                "genre",
                "director",
                "rating",
                "originalLanguage",
                "releaseYear",
                "audienceScore",
                "tomatoMeter",
                "hybrid_score"
            ]
        ].to_string(index=False)
    )

    return final_output



#=========================================================
#Tests
#==========================================================
integrated_movie_recommender(
    search_input="Batman",
    top_n=10,
    sort_by="Hybrid Score", #Tomato Meter, Audience Score, Release Year, Hybrid Score
    ascending=True
)



🔍 SEARCH QUERY : Batman
📌 SEARCH TYPE : TITLE

🎬 TOP RESULT
----------------------------------------------------------------------
 title                      genre  director  rating originalLanguage  releaseYear
batman action, adventure, fantasy timburton Unknown          english         1989

🔥 HYBRID FILTERED RESULTS ONLY
----------------------------------------------------------------------
                                        title                                 genre         director  rating originalLanguage  releaseYear  audienceScore  tomatoMeter  hybrid_score
                                     big fish                                 drama        timburton   PG-13          english         2004           89.0         75.0      0.645582
the batman (an evening with batman and robin)                               unknown   lamberthillyer Unknown          unknown         1900           60.0         74.0      0.648174
                     batman: the killing joke action, adve

,id,title,genre,director,rating,originalLanguage,releaseDateTheaters,runtimeMinutes,audienceScore,tomatoMeter,...,bayesianAudienceScore,bayesianTomatoMeter,finalScore,rating_numeric,rating_label,combined_features,content_score,svd_score,svd_score_norm,hybrid_score
0,1127787-big_fish,big fish,drama,timburton,PG-13,english,2004-01-09,125.0,89.0,75.0,...,88.243301,74.867807,82.893104,2.0,PG-13,drama timburton PG-13 english big fish,0.409304,100,1.0,0.645582
1,10005205-batman,the batman (an evening with batman and robin),unknown,lamberthillyer,Unknown,unknown,NaT,104.0,60.0,74.0,...,60.846194,71.098175,64.946986,-1.0,Unknown,unknown lamberthillyer Unknown unknown the bat...,0.413623,100,1.0,0.648174
2,batman_the_killing_joke,batman: the killing joke,"action, adventure, fantasy, animation",samliu,R,english,2016-07-25,76.0,50.0,36.0,...,51.309207,40.015400,46.791684,3.0,R,"action, adventure, fantasy, animation samliu R...",0.415590,100,1.0,0.649354
3,the_batman,the batman,"action, adventure, crime, drama",mattreeves,PG-13,english,2022-03-04,176.0,87.0,85.0,...,86.697995,84.826431,85.949369,2.0,PG-13,"action, adventure, crime, drama mattreeves PG-...",0.425238,100,1.0,0.655143
4,batman_begins,batman begins,"action, fantasy, adventure",christophernolan,PG-13,english,2005-06-15,140.0,94.0,84.0,...,93.324553,83.715018,89.480739,2.0,PG-13,"action, fantasy, adventure christophernolan PG...",0.451989,100,1.0,0.671194
5,1077027-batman_and_robin,batman & robin,"action, adventure, fantasy",joelschumacher,PG-13,english,1997-06-20,130.0,16.0,12.0,...,18.735046,15.523085,17.450262,2.0,PG-13,"action, adventure, fantasy joelschumacher PG-1...",0.463920,100,1.0,0.678352
6,dark-shadows-2010,dark shadows,"comedy, fantasy",timburton,PG-13,english,2012-05-11,112.0,46.0,35.0,...,46.339961,35.789458,42.119760,2.0,PG-13,"comedy, fantasy timburton PG-13 english dark s...",0.478915,100,1.0,0.687349
7,frankenweenie-1992,frankenweenie,fantasy,timburton,Unknown,english,1984-12-14,27.0,74.0,74.0,...,64.346194,71.098175,67.046986,-1.0,Unknown,fantasy timburton Unknown english frankenweenie,0.503798,100,1.0,0.702279
8,batman_forever,batman forever,"action, adventure, fantasy",joelschumacher,Unknown,english,1995-06-16,121.0,32.0,39.0,...,34.427355,41.594242,37.294110,-1.0,Unknown,"action, adventure, fantasy joelschumacher Unkn...",0.506088,100,1.0,0.703653
9,batman_returns,batman returns,"action, adventure, fantasy",timburton,Unknown,english,1992-06-19,126.0,73.0,81.0,...,72.234081,80.298768,75.459956,-1.0,Unknown,"action, adventure, fantasy timburton Unknown e...",0.860492,100,1.0,0.916295
